# Optimización definitiva de Crop8

Optuna ajusta gamma y delta entre 0 y 2. Los parámetros de la máscara se fijan en bias = 0.22 y slope = 22. Cada dataset tiene un estudio para maximizar acc_both y otro para maximizar SSIM.

Se usan los CVAE de dimensión intermedia 256 y dimensión latente 64 del cuaderno de Crop7. Cada estudio evalúa 100 pruebas de 30 iteraciones sobre el mismo lote de 1.000 mezclas de entrenamiento, con semilla 3333.

Los estudios se guardan en `results_crop8/<dataset>/<objetivo>/Crop8.sqlite3`. Al volver a ejecutar un estudio, se completan las pruebas pendientes.

## Configuración y evaluación

In [1]:
from pathlib import Path
import sys

import numpy as np
import optuna

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "project" / "CROP" / "models").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abrí el notebook desde el repositorio.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENTS_DIR = PROJECT_ROOT / "project" / "definitive_experiments"
RESULTS_DIR = EXPERIMENTS_DIR / "results_crop8"
SEED = 3333
LAT, INTER = 64, 256
N_IMAGES = 1000
N_TRIALS = 100
STUDY_ITERATIONS = 30

DATASETS = {"fashion": "Fashion-MNIST", "mnist": "MNIST"}
SEARCH_SPACE = {
    "gamma": [0.0, 2.0, False],
    "delta": [0.0, 2.0, False],
}
FIXED_PARAMS = {
    "bias": 0.22,
    "slope": 22,
    "alpha_1": -2,
    "alpha_2": -22,
    "alpha_mix": 0.5,
    "beta": 1,
}


def result_dir(dataset, objective_metric):
    return RESULTS_DIR / dataset / objective_metric


def load_inputs(dataset, latent_dim, intermediate_dim):
    import tensorflow as tf
    from project.CROP.models.crop8 import Crop8
    from project.loader import Loader

    tf.keras.utils.set_random_seed(SEED)
    return (
        Loader.data(dataset),
        Loader.cvae(model_name=f"int_{intermediate_dim}_lat_{latent_dim}_{dataset}.keras"),
        Loader.predictor(dataset=dataset),
        Crop8,
    )


def sample_batch(data, split, seed, n_images):
    rng = np.random.default_rng(seed)
    x0, x1 = data[f"x_{split}"], data[f"x_{split}_1"]
    y0, y1 = data[f"y_{split}"], data[f"y_{split}_1"]
    idx0 = rng.choice(len(x0), size=n_images, replace=False)
    idx1 = rng.choice(len(x1), size=n_images, replace=False)
    return x0[idx0], x1[idx1], y0[idx0], y1[idx1]


def evaluate(inputs, batch, params, seed, iterations):
    import tensorflow as tf

    data, cvae, predictor, model_class = inputs
    tf.keras.utils.set_random_seed(seed)
    model = model_class(cvae=cvae, predictor=predictor)
    return model.unmix(
        *batch,
        params=params,
        iterations=iterations,
        labels=data["labels"],
        show_metrics=False,
    )


def scalar_metric(metrics, key):
    value = metrics[key]
    return float(value[0] if isinstance(value, (tuple, list)) else value)

/home/galsan/venvs/tensorflow-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Estudios de Optuna

In [2]:
def study_location(dataset, objective_metric):
    path = result_dir(dataset, objective_metric) / "Crop8.sqlite3"
    name = f"{dataset}_{objective_metric}_Crop8"
    return path, f"sqlite:///{path.as_posix()}", name


def load_study(dataset, objective_metric):
    path, storage, name = study_location(dataset, objective_metric)
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el estudio en {path}")
    return optuna.load_study(storage=storage, study_name=name)


def optimize_crop8(dataset, objective_metric, inputs, train_batch):
    path, storage, name = study_location(dataset, objective_metric)
    path.parent.mkdir(parents=True, exist_ok=True)
    study = optuna.create_study(
        storage=storage,
        study_name=name,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        load_if_exists=True,
    )
    config = {
        "seed": SEED,
        "images": N_IMAGES,
        "iterations": STUDY_ITERATIONS,
        "latent_dim": LAT,
        "intermediate_dim": INTER,
        "fixed_params": FIXED_PARAMS,
        "search_space": SEARCH_SPACE,
    }
    saved_config = study.user_attrs.get("config")
    if saved_config is None and not study.get_trials(deepcopy=False):
        study.set_user_attr("config", config)
    elif saved_config != config:
        raise ValueError(f"La configuración guardada en {path} difiere de la actual.")

    def objective(trial):
        params = {
            key: trial.suggest_float(key, low, high, log=log)
            for key, (low, high, log) in SEARCH_SPACE.items()
        }
        effective_params = {**FIXED_PARAMS, **params}
        metrics = evaluate(inputs, train_batch, effective_params, SEED, STUDY_ITERATIONS)
        score = scalar_metric(metrics, objective_metric)
        if not np.isfinite(score):
            raise ValueError(f"Métrica no finita en {name}: {score}")
        trial.set_user_attr("effective_params", effective_params)
        for key in ("acc_both", "ssim", "recon_bpsnr"):
            value = scalar_metric(metrics, key)
            trial.set_user_attr(key, value if np.isfinite(value) else None)
        return score

    completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,), deepcopy=False)
    remaining = max(0, N_TRIALS - len(completed))
    if remaining:
        if completed:
            # Evitar repetir las muestras iniciales al retomar el estudio.
            study.sampler = optuna.samplers.TPESampler(seed=SEED + len(completed))
        study.optimize(objective, n_trials=remaining, n_jobs=1)

    print(f"{name} · {study.best_value:.6g} · {study.best_params}", flush=True)
    return study


def plot_study(study, dataset, objective_metric):
    from optuna.visualization import plot_slice

    fig = plot_slice(study, params=list(SEARCH_SPACE),
                     target_name=objective_metric)
    fig.update_layout(title=f"{DATASETS[dataset]} · Crop8 · {objective_metric}")
    fig.show()


def run_experiment(dataset, objective_metric):
    inputs = load_inputs(dataset, LAT, INTER)
    train_batch = sample_batch(inputs[0], "train", SEED, N_IMAGES)
    return optimize_crop8(dataset, objective_metric, inputs, train_batch)

## Fashion-MNIST

In [3]:
fashion_acc = run_experiment("fashion", "acc_both")

I0000 00:00:1795849725.540082  283076 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795849725.591807  283076 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795849726.895852  283076 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Usando fashion como dataset


I0000 00:00:1795849728.457888  283076 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1042 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6
[I 2026-11-28 04:08:50,032] A new study created in RDB with name: fashion_acc_both_Crop8
I0000 00:00:1795849730.694567  283076 cuda_dnn.cc:461] Loaded cuDNN version 92801
[I 2026-11-28 04:08:54,523] Trial 0 finished with value: 0.59 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547}. Best is trial 0 with value: 0.59.
[I 2026-11-28 04:08:56,289] Trial 1 finished with value: 0.451 and parameters: {'gamma': 0.9722425581148704, 'delta': 0.9996623554960684}. Best is trial 0 with value: 0.59.
[I 2026-11-28 04:08:57,960] Trial 2 finished with value: 0.546 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138}. Best is trial 0 with value: 0.59.
[I 2026-11-28 04:08:59,605] Trial 3 finished with value: 0.617 and param

fashion_acc_both_Crop8 · 0.719 · {'gamma': 0.025676863485455503, 'delta': 0.31640031629658794}


In [4]:
fashion_ssim = run_experiment("fashion", "ssim")

Usando fashion como dataset


[I 2026-11-28 04:11:27,934] A new study created in RDB with name: fashion_ssim_Crop8
[I 2026-11-28 04:11:29,490] Trial 0 finished with value: 0.6764737367630005 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547}. Best is trial 0 with value: 0.6764737367630005.
[I 2026-11-28 04:11:31,055] Trial 1 finished with value: 0.6026397347450256 and parameters: {'gamma': 0.9722425581148704, 'delta': 0.9996623554960684}. Best is trial 0 with value: 0.6764737367630005.
[I 2026-11-28 04:11:32,525] Trial 2 finished with value: 0.657395601272583 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138}. Best is trial 0 with value: 0.6764737367630005.
[I 2026-11-28 04:11:34,098] Trial 3 finished with value: 0.6814298629760742 and parameters: {'gamma': 1.2255897813211882, 'delta': 0.23930480679487687}. Best is trial 3 with value: 0.6814298629760742.
[I 2026-11-28 04:11:35,677] Trial 4 finished with value: 0.6253678202629089 and parameters: {'gamma': 1.4244045932

fashion_ssim_Crop8 · 0.739118 · {'gamma': 0.0007254477362136462, 'delta': 0.37541147832186705}


## MNIST

In [5]:
mnist_acc = run_experiment("mnist", "acc_both")

Usando mnist como dataset


[I 2026-11-28 04:14:02,430] A new study created in RDB with name: mnist_acc_both_Crop8
[I 2026-11-28 04:14:04,140] Trial 0 finished with value: 0.779 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547}. Best is trial 0 with value: 0.779.
[I 2026-11-28 04:14:05,795] Trial 1 finished with value: 0.525 and parameters: {'gamma': 0.9722425581148704, 'delta': 0.9996623554960684}. Best is trial 0 with value: 0.779.
[I 2026-11-28 04:14:07,480] Trial 2 finished with value: 0.73 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138}. Best is trial 0 with value: 0.779.
[I 2026-11-28 04:14:09,076] Trial 3 finished with value: 0.805 and parameters: {'gamma': 1.2255897813211882, 'delta': 0.23930480679487687}. Best is trial 3 with value: 0.805.
[I 2026-11-28 04:14:10,615] Trial 4 finished with value: 0.572 and parameters: {'gamma': 1.4244045932667562, 'delta': 0.7451210841006557}. Best is trial 3 with value: 0.805.
[I 2026-11-28 04:14:12,156] Trial 5 finish

mnist_acc_both_Crop8 · 0.881 · {'gamma': 0.0007254477362136462, 'delta': 0.37541147832186705}


In [6]:
mnist_ssim = run_experiment("mnist", "ssim")

Usando mnist como dataset


[I 2026-11-28 04:16:36,164] A new study created in RDB with name: mnist_ssim_Crop8
[I 2026-11-28 04:16:37,849] Trial 0 finished with value: 0.7565648555755615 and parameters: {'gamma': 1.5031512144089574, 'delta': 0.21850255077708547}. Best is trial 0 with value: 0.7565648555755615.
[I 2026-11-28 04:16:39,290] Trial 1 finished with value: 0.6227676272392273 and parameters: {'gamma': 0.9722425581148704, 'delta': 0.9996623554960684}. Best is trial 0 with value: 0.7565648555755615.
[I 2026-11-28 04:16:40,906] Trial 2 finished with value: 0.7233947515487671 and parameters: {'gamma': 1.3142199980396865, 'delta': 0.4717694224816138}. Best is trial 0 with value: 0.7565648555755615.
[I 2026-11-28 04:16:42,541] Trial 3 finished with value: 0.7715882658958435 and parameters: {'gamma': 1.2255897813211882, 'delta': 0.23930480679487687}. Best is trial 3 with value: 0.7715882658958435.
[I 2026-11-28 04:16:44,058] Trial 4 finished with value: 0.6468777060508728 and parameters: {'gamma': 1.42440459326

mnist_ssim_Crop8 · 0.853179 · {'gamma': 0.0005674996032943125, 'delta': 0.059651514000241636}


## Resultados guardados y evaluación en prueba

Elegí un estudio guardado para ver sus mejores parámetros y los gráficos de cortes de Optuna. La última celda evalúa esos parámetros en un lote independiente de prueba con la misma cantidad de mezclas e iteraciones del estudio.

In [10]:
dataset, objective_metric = "mnist", "acc_both"
""

study = load_study(dataset, objective_metric)
print(f"Crop8 · {study.best_value:.6g} · {study.best_params}")
plot_study(study, dataset, objective_metric)

Crop8 · 0.881 · {'gamma': 0.0007254477362136462, 'delta': 0.37541147832186705}


In [8]:
config = study.user_attrs["config"]
test_seed = config["seed"] + 1
inputs = load_inputs(dataset, config["latent_dim"], config["intermediate_dim"])
test_batch = sample_batch(inputs[0], "test", test_seed, config["images"])
params = {**config["fixed_params"], **study.best_params}
test_metrics = evaluate(inputs, test_batch, params, test_seed, config["iterations"])

for metric in ("acc_both", "ssim", "recon_bpsnr"):
    print(f"{metric} = {scalar_metric(test_metrics, metric):.6g}")

Usando fashion como dataset
acc_both = 0.695
ssim = 0.738976
recon_bpsnr = 20.061
